# Qwen3-VL — Table Structure Reconstruction

Feasibility spike. **Demo: Wednesday 22 July 2026.**

Question: can Qwen3-VL reconstruct hard table structure (merged cells,
rowspan/colspan, mixed bordered/borderless) well enough to justify committing
company resources?

Logic lives in `src/`; this notebook only drives it. **Read `RUNBOOK.md` first** —
it has the step-by-step order, timings, and what to do when a step fails.

| Day | Output | Gate |
|---|---|---|
| 1 | Corpus + zero-shot baseline + side-by-side page | demo-able artifact exists |
| 2 | Prompt iteration | best prompt chosen, measured |
| 3 | LoRA fine-tune | adapter trained, evaluated |
| 4 | Comparison table + final demo page | dry-run passes |

**Primary metric is TEDS-Struct** — structure only, text ignored. Plain TEDS
would let OCR errors masquerade as structural ones.

**One model in VRAM at a time.** Each stage calls `.close()` before the next
loads. Skipping that stacks two 4B models on one card and OOMs *after* the
training run rather than before it.

## Setup

Run once per fresh runtime. On Colab this needs a restart afterwards.

In [ ]:
# On Lightning AI / Colab only. Skip locally.
# !pip install -q -r requirements-base.txt
# !pip install -q -r requirements-gpu.txt
# Colab: restart the runtime here before continuing.

In [ ]:
import sys
from pathlib import Path

# Repo root on sys.path so `src` imports work from the notebook.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CORPUS = ROOT / "data" / "corpus"
RUNS = ROOT / "data" / "runs"
DEMO = ROOT / "data" / "demo"
for d in (CORPUS, RUNS, DEMO):
    d.mkdir(parents=True, exist_ok=True)

from src.model.inference import gpu_report, free_memory
print(gpu_report())

---
## Day 1 — Corpus

FinTabNet (financial tables from S&P 500 annual reports) rather than PubTabNet:
borderless-heavy, hierarchical, and closer to invoice layouts than biomedical
tables.

Difficulty is scored from ground-truth HTML, then the **hardest N are selected by
rank**. An absolute threshold has to be guessed per dataset and guesses badly —
measured on 3,000 FinTabNet tables the distribution is p50=0.12, p90=0.23, so a
0.55 cutoff matched 1 table in 3,000.

Only HTML is scanned; images download only for tables that are kept. The bulk
train split is 10.28 GB — streaming it stalls on a home connection.

In [ ]:
from src.data.loader import build_corpus

# ~15-20 min for 500/100. Raise max_scanned for a harder corpus.
counts = build_corpus(CORPUS, n_train=500, n_eval=100, max_scanned=20000)
counts

In [ ]:
from src.data.loader import load_manifest

train_records = load_manifest(CORPUS / "train")
eval_records = load_manifest(CORPUS / "eval")

print(f"train {len(train_records)} | eval {len(eval_records)}")
for label in ("easy", "medium", "hard"):
    n = sum(1 for r in eval_records if r.complexity.bin == label)
    spans = [r.complexity.n_spanning for r in eval_records if r.complexity.bin == label]
    if spans:
        print(f"  {label:<7} n={n:<4} mean merged cells={sum(spans)/len(spans):.1f}")

### Zero-shot baseline

The number the whole project is otherwise being planned without.

`use_unsloth=True` here matters for a reason that is easy to miss: the fine-tuned
model on Day 3 is loaded through Unsloth, and Unsloth's 4-bit quantization is not
bit-identical to plain bitsandbytes. Load the baseline the other way and part of
your measured "fine-tuning gain" is a quantization artifact. **Same loader for
both arms, always.**

In [ ]:
from src.model.inference import TableReconstructor, MODEL_4B, MODEL_8B

model = TableReconstructor(model_id=MODEL_4B, load_in_4bit=True, use_unsloth=True)
print(gpu_report())

# `full` mode for the demo visuals (rendered tables look real);
# `structure` mode is what gets trained and scored.
preds = model.predict_many([r.image_path for r in eval_records], mode="structure")
predictions = {p.uid: p.html for p in preds}

failed = sum(1 for p in preds if not p.html.strip())
print(f"{len(preds)} predictions, {failed} unparseable")

In [ ]:
from src.eval.runner import evaluate_predictions, save_run

results, summary = evaluate_predictions(eval_records, predictions, "zeroshot-4b")
save_run(results, summary, RUNS)

print(f"TEDS-Struct  {summary.mean_teds:.4f}  95% CI [{summary.ci_low:.3f}, {summary.ci_high:.3f}]")
print(f"span recall  {summary.mean_span_recall:.4f}")
print(f"parse fails  {summary.parse_failures}")
for label, stats in summary.by_bin.items():
    print(f"  {label:<7} n={stats['n']:<4} {stats['mean']:.4f}")

### Side-by-side page

The demo centerpiece. Self-contained — opens offline, nothing runs live.

In [ ]:
from src.demo.render import render_comparison
from src.eval.runner import to_cases

# hardest_first shows the failures too; a reel with no failures invites
# the question of what was left out.
path = render_comparison(
    to_cases(results, limit=20, hardest_first=True),
    DEMO / "day1_zeroshot.html",
    title="Qwen3-VL Zero-Shot — Hard FinTabNet Tables",
)
print(path)

### Optional: 8B zero-shot ceiling

Inference needs far less VRAM than training, so this datapoint is cheap. Skip it
if Day 1 is running late — it is upside, not a gate.

Close the 4B first. Two models on one card is an OOM.

In [ ]:
model.close()
print(gpu_report())

model_8b = TableReconstructor(model_id=MODEL_8B, load_in_4bit=True, use_unsloth=True)
preds_8b = model_8b.predict_many([r.image_path for r in eval_records], mode="structure")
results_8b, summary_8b = evaluate_predictions(
    eval_records, {p.uid: p.html for p in preds_8b}, "zeroshot-8b"
)
save_run(results_8b, summary_8b, RUNS)

from src.eval.runner import compare_runs
print(compare_runs(summary, summary_8b))

model_8b.close()

---
## Day 2 — Prompt iteration

Costs minutes and zero GPU-hours. Do this before spending a training run — many
"we need fine-tuning" problems are prompt and output-format problems.

**Define variants as strings here, not by editing `src/model/prompts.py`.** The
module is already imported; editing the file and re-running a cell scores the
*old* prompt again, which reads as "prompt engineering did nothing."

Iterate on a 30-table slice, then confirm the winner on the full eval set. Each
variant over 100 tables costs real minutes and the ranking rarely changes.

In [ ]:
from src.eval.runner import compare_runs

model = TableReconstructor(model_id=MODEL_4B, load_in_4bit=True, use_unsloth=True)

PROMPT_V2 = (
    "Reconstruct this table's HTML structure. Work row by row from the top. "
    "A cell that visually spans several rows needs rowspan; one spanning several "
    "columns needs colspan. Every row must account for the same total number of "
    "columns. Leave all cells empty -- emit no cell text. Output raw HTML "
    "starting with <table>. No markdown fences, no explanation."
)

probe = eval_records[:30]
probe_preds = model.predict_many(
    [r.image_path for r in probe], mode="structure", instruction=PROMPT_V2
)
probe_results, probe_summary = evaluate_predictions(
    probe, {p.uid: p.html for p in probe_preds}, "probe-v2"
)

# Baseline on the same 30 so the comparison is like-for-like.
base_probe = [r for r in results if r.uid in {p.uid for p in probe_preds}]
from src.eval.runner import summarize_run
print(compare_runs(summarize_run(base_probe, "probe-v1"), probe_summary))

In [ ]:
# Confirm the winner on the full eval set, then keep it in a variable --
# Day 3 must train with the SAME prompt it generates with.
BEST_PROMPT = PROMPT_V2   # or None to keep the default

preds_v2 = model.predict_many(
    [r.image_path for r in eval_records], mode="structure", instruction=BEST_PROMPT
)
results_v2, summary_v2 = evaluate_predictions(
    eval_records, {p.uid: p.html for p in preds_v2}, "zeroshot-4b-prompt-v2"
)
save_run(results_v2, summary_v2, RUNS)
print(compare_runs(summary, summary_v2))

---
## Day 3 — LoRA fine-tune

Qwen3-VL-4B, not 8B: under a deadline **iteration count beats parameter count**.
A 4B trains in ~2h leaving room for a second attempt; an 8B that takes 8h and
OOMs twice leaves nothing on Tuesday night.

Checkpoints every 50 steps and resumes automatically — free-tier runtimes
disconnect.

In [ ]:
# Free the inference model before training. This is the OOM that bites people:
# it fires 40 minutes in, after the run looked healthy.
model.close()
print(gpu_report())

In [ ]:
from src.train.lora import TrainConfig, train

cfg = TrainConfig(
    model_id=MODEL_4B,
    output_dir=str(ROOT / "outputs" / "lora-4b-structure"),
    mode="structure",
    instruction=BEST_PROMPT,   # must match what inference will use
    epochs=2,
    max_seq_length=4096,
)

# Safe to re-run after a disconnect; picks up from the last checkpoint.
ft_model, ft_processor, trainer = train(train_records, cfg, resume=True)

In [ ]:
# Reload from the saved adapter rather than reusing the in-memory model: it
# proves the adapter actually persisted, which is what Wednesday depends on.
del ft_model, ft_processor, trainer
free_memory()
print(gpu_report())

tuned = TableReconstructor(
    model_id=MODEL_4B,
    load_in_4bit=True,
    use_unsloth=True,
    adapter_path=str(ROOT / "outputs" / "lora-4b-structure" / "adapter"),
)
preds_ft = tuned.predict_many(
    [r.image_path for r in eval_records], mode="structure", instruction=BEST_PROMPT
)

results_ft, summary_ft = evaluate_predictions(
    eval_records, {p.uid: p.html for p in preds_ft}, "lora-4b"
)
save_run(results_ft, summary_ft, RUNS)
print(compare_runs(summary, summary_ft))

---
## Day 4 — Demo assembly

The comparison table carries two of the three demo goals at once: *fine-tuning
helps* and *it handles the hard cases*.

If the confidence intervals overlap, `compare_runs` says NOT SIGNIFICANT — that
is the honest reading at ~100 eval tables, and presenting the gap anyway is a
claim that will not survive a larger evaluation later.

In [ ]:
# Rebuilds from saved JSON -- no GPU needed, so this cell also works on the Mac.
from src.eval.runner import load_run

results, summary = load_run(RUNS / "zeroshot-4b.json")
results_ft, summary_ft = load_run(RUNS / "lora-4b.json")
print(compare_runs(summary, summary_ft))

In [ ]:
# Final page. `full` mode so rendered tables carry text and look real --
# label clearly that the *metric* is structure-only.
demo_records = [r for r in eval_records if r.complexity.bin == "hard"][:10]
demo_preds = tuned.predict_many(
    [r.image_path for r in demo_records], mode="full", instruction=None
)

demo_results, _ = evaluate_predictions(
    demo_records, {p.uid: p.html for p in demo_preds}, "demo-visuals"
)
path = render_comparison(
    to_cases(demo_results, hardest_first=False),
    DEMO / "final_demo.html",
    title="Qwen3-VL Fine-Tuned — Hard Financial Tables",
    subtitle="TEDS-Struct measures structure only; cell text shown for legibility.",
)
print(f"open this at the demo: {path}")

### Pre-flight checklist

- [ ] `pytest tests/ -q` passes
- [ ] `assert_no_leakage` passed on the built corpus
- [ ] Notebook runs top-to-bottom on a **fresh** runtime
- [ ] `final_demo.html` opens with the network off
- [ ] Every output pre-rendered — nothing runs live at the demo
- [ ] Both arms loaded the same way (`use_unsloth=True`), so the delta is the
      adapter and not a quantization difference
- [ ] Know the honest answer to "how much of the gain is output-format
      alignment rather than better table reasoning?"